# MLPerf Vision Benchmark
Benchmarks image classification throughput and latency using **torchvision** pretrained models.
The model is downloaded automatically from PyTorch Hub on first run and cached locally.
Uses fake ImageNet images by default — no dataset download required.

**Run cells top to bottom. Edit only Cell 2.**

## Cell 1 — GPU Verification

In [ ]:
import torch
print('=== GPU Status ===')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i} : {p.name}  {p.total_memory/1e9:.1f} GB VRAM')
    a = torch.rand(1024,1024,device='cuda') @ torch.rand(1024,1024,device='cuda')
    torch.cuda.synchronize()
    print('Smoke test : PASSED')
else:
    print('WARNING: No GPU detected — benchmark will run on CPU (much slower).')

## Cell 2 — Configuration
Edit this cell to select the model and run settings.

In [ ]:
# ── Edit these ──────────────────────────────────────────────────────────────
MODEL_NAME   = 'resnet50'      # 'resnet50' | 'resnet18' | 'resnet101' | 'vgg16'
BATCH_SIZE   = 32              # images per inference call
NUM_WARMUP   = 5               # warmup batches (not measured)
NUM_BENCH    = 50              # benchmark batches
RUN_ACCURACY = True            # compute top-1 accuracy (always 0 on fake data)
USE_FP16     = True            # half-precision inference (faster on modern GPUs)

# Optional: point to a real ImageNet-style directory (subfolders per class).
# Leave as None to use auto-generated fake images.
DATA_DIR = None
# DATA_DIR = '/path/to/imagenet/val'
# ────────────────────────────────────────────────────────────────────────────

import os, json
LOG_DIR = '/tmp/mlperf_vision_results'
os.makedirs(LOG_DIR, exist_ok=True)

print(f'Model       : {MODEL_NAME}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Batches     : {NUM_BENCH}  (+ {NUM_WARMUP} warmup)')
print(f'Total imgs  : {(NUM_WARMUP + NUM_BENCH) * BATCH_SIZE}')
print(f'FP16        : {USE_FP16}')
print(f'Data source : {DATA_DIR or "fake ImageNet (generated)"}')

## Cell 3 — Download Model
Loads the pretrained model from `torchvision.models` (PyTorch Hub).
Weights are cached under `~/.cache/torch/hub/` — skipped on subsequent runs.

In [ ]:
import torch, torchvision.models as tvm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Map name → torchvision weights enum
_WEIGHTS = {
    'resnet18':  tvm.ResNet18_Weights.IMAGENET1K_V1,
    'resnet50':  tvm.ResNet50_Weights.IMAGENET1K_V2,
    'resnet101': tvm.ResNet101_Weights.IMAGENET1K_V2,
    'vgg16':     tvm.VGG16_Weights.IMAGENET1K_V1,
}

if MODEL_NAME not in _WEIGHTS:
    raise ValueError(f'Unknown model "{MODEL_NAME}". Choose from: {list(_WEIGHTS)}')

weights = _WEIGHTS[MODEL_NAME]
print(f'Downloading / loading {MODEL_NAME} ({weights}) ...')

model = getattr(tvm, MODEL_NAME)(weights=weights)
model.eval()
model = model.to(DEVICE)
if USE_FP16 and DEVICE == 'cuda':
    model = model.half()

# Image pre-processing transform from the model's metadata
preprocess = weights.transforms()

print(f'Model ready on {DEVICE}' + (' (FP16)' if USE_FP16 and DEVICE=='cuda' else ''))

## Cell 4 — Prepare Dataset
Uses real ImageNet images if `DATA_DIR` is set, otherwise generates a fake dataset
of random tensors (same shape as ImageNet). Fake data is reproducible and needs no download.

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset, TensorDataset
import torchvision.transforms as T

total_images = (NUM_WARMUP + NUM_BENCH) * BATCH_SIZE

if DATA_DIR and os.path.isdir(DATA_DIR) and len(os.listdir(DATA_DIR)) > 0:
    # Real ImageNet-style directory
    from torchvision.datasets import ImageFolder
    dataset  = ImageFolder(DATA_DIR, transform=preprocess)
    # Subsample if large
    indices  = list(range(min(total_images, len(dataset))))
    dataset  = torch.utils.data.Subset(dataset, indices)
    loader   = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)
    print(f'Real images : {DATA_DIR}  ({len(dataset)} samples)')
else:
    # Fake ImageNet: random tensors, deterministic seed
    print('Generating fake ImageNet tensors (reproducible) ...')
    torch.manual_seed(42)
    imgs   = torch.randn(total_images, 3, 224, 224)
    labels = torch.randint(0, 1000, (total_images,))
    if USE_FP16 and DEVICE == 'cuda':
        imgs = imgs.half()
    dataset = TensorDataset(imgs, labels)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0)
    print(f'Fake images : {total_images} tensors  (3x224x224, random)')

batches = list(loader)
print(f'Batches prepared: {len(batches)}')

## Cell 5 — Run Inference Benchmark
Measures throughput (images/sec) and per-batch latency.
Warmup batches prime the GPU and are excluded from measurements.

In [ ]:
import time, json, statistics
import torch

correct = 0
total   = 0
latencies_ms = []

# ── Warmup ──────────────────────────────────────────────────────────────────
print(f'Warming up ({NUM_WARMUP} batches) ...')
with torch.no_grad():
    for imgs, labels in batches[:NUM_WARMUP]:
        imgs = imgs.to(DEVICE)
        if USE_FP16 and DEVICE == 'cuda' and imgs.dtype != torch.float16:
            imgs = imgs.half()
        _ = model(imgs)
        if DEVICE == 'cuda': torch.cuda.synchronize()

# ── Benchmark ───────────────────────────────────────────────────────────────
print(f'Benchmarking ({NUM_BENCH} batches) ...')
bench_batches = batches[NUM_WARMUP:NUM_WARMUP + NUM_BENCH]

t_total_start = time.perf_counter()

with torch.no_grad():
    for imgs, labels in bench_batches:
        imgs   = imgs.to(DEVICE)
        labels = labels.to(DEVICE)
        if USE_FP16 and DEVICE == 'cuda' and imgs.dtype != torch.float16:
            imgs = imgs.half()

        t0  = time.perf_counter()
        out = model(imgs)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        t1  = time.perf_counter()

        latencies_ms.append((t1 - t0) * 1000)

        if RUN_ACCURACY:
            preds   = out.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

t_total_end = time.perf_counter()

# ── Metrics ─────────────────────────────────────────────────────────────────
elapsed       = t_total_end - t_total_start
total_images  = NUM_BENCH * BATCH_SIZE
throughput    = total_images / elapsed
mean_lat      = statistics.mean(latencies_ms)
p50_lat       = statistics.median(latencies_ms)
p99_lat       = sorted(latencies_ms)[int(len(latencies_ms) * 0.99)]

results = {
    'model':           MODEL_NAME,
    'batch_size':      BATCH_SIZE,
    'num_batches':     NUM_BENCH,
    'total_images':    total_images,
    'elapsed_s':       round(elapsed, 3),
    'throughput_img_s':round(throughput, 2),
    'latency_mean_ms': round(mean_lat, 2),
    'latency_p50_ms':  round(p50_lat, 2),
    'latency_p99_ms':  round(p99_lat, 2),
    'device':          DEVICE,
    'fp16':            (USE_FP16 and DEVICE == 'cuda'),
}
if RUN_ACCURACY and total > 0:
    results['top1_accuracy'] = round(correct / total, 4)

out_path = f'{LOG_DIR}/results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved: {out_path}')

## Cell 6 — Results

In [ ]:
import json

out_path = f'{LOG_DIR}/results.json'
if not os.path.exists(out_path):
    print('No results file — check Cell 5 output.')
else:
    r = json.load(open(out_path))
    print('=== Vision Benchmark Results ===')
    print(f"  Model             : {r['model']}")
    print(f"  Device            : {r['device']}" + (' (FP16)' if r['fp16'] else ''))
    print(f"  Batch size        : {r['batch_size']}")
    print(f"  Total images      : {r['total_images']}")
    print(f"  Elapsed           : {r['elapsed_s']} s")
    print(f"  Throughput        : {r['throughput_img_s']} img/s")
    print(f"  Latency (mean)    : {r['latency_mean_ms']} ms / batch")
    print(f"  Latency (p50)     : {r['latency_p50_ms']} ms")
    print(f"  Latency (p99)     : {r['latency_p99_ms']} ms")
    if 'top1_accuracy' in r:
        note = '  (fake data → random labels, accuracy is expected ~0.1%)' if not DATA_DIR else ''
        print(f"  Top-1 accuracy    : {r['top1_accuracy']:.2%}{note}")
    print()
    print('Full results:')
    print(json.dumps(r, indent=2))